# tp-05f 精度向上の実験（NN の拡張・相対化の拡張）

**実行場所: ホスト PC（stg）。** tp-05d で最良だった小さな NN（`nn_tuned`）と木モデルを基準に、精度向上の余地を 1 つずつ試す。**採否は valid 2022〜2024 だけで決める**（test 2025 は既に一度見たので使わない）。

| 案 | 内容 |
|---|---|
| nn_emb | 騎手・調教師・種牡馬・母父の埋め込み（学習で 30 回以上出る値だけ個別、残りは「その他」） |
| nn_attn1 / nn_attn2 | 同じレースの馬どうしの注意機構（Transformer の encoder 1 層 / 2 層） |
| nn_emb_attn1 | 埋め込み + 注意機構 |
| nn_c2 | 相対化を広い列（展開・レーティング・ペース系）に拡張した特徴量（C2）を NN に |
| lgb_c2 / xgb_c2 | 同じ C2 特徴量を LightGBM・XGBoost（調整後パラメータ）に |

基準は tp-05d の `nn_tuned` / `lgb_tuned` / `xgb_tuned`（保存済み OOF）。採否: p_win の LogLoss の差の 95% 信頼区間の上限が 0 未満で、連対・複勝が悪化しない。条件は tp-05b と同じ（3 fold × 3 seed）。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "6")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc
import numpy as np
import pandas as pd
import mlflow
import psutil

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import tp05_data, zoo, zoo_runner, win_models as wm
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "improve_f"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = detect_compute(reserve_cores=6).threads
SEEDS = [0, 1, 2]
mlflow.set_experiment("keiba-tp-model-zoo")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
IDS = ["jockey_id", "trainer_id", "sire", "dam_sire"]
X1 = df[D["FEATS"]["base+S+C1"]]
X1_ids = pd.concat([X1, df[IDS].astype(str)], axis=1)
X2 = df[D["FEATS"]["base+S+C1+C2"]]
X2_ids = pd.concat([X2, df[IDS].astype(str)], axis=1)
nn_p = zoo.tuned_params("nn", best["nn"]["params"])
print(f"行 {len(df):,} / C1 {X1.shape[1]} 列 / C2 {X2.shape[1]} 列 / threads {THREADS} / mem {psutil.Process().memory_info().rss / 1e9:.1f}GB")

In [ ]:
EXPS = {  # 名前: (recipe, X, params, 基準 OOF)
    "nn_emb": ("nn", X1_ids, {**nn_p, "id_cols": IDS}, "nn_tuned"),
    "nn_attn1": ("nn", X1, {**nn_p, "attn_layers": 1}, "nn_tuned"),
    "nn_attn2": ("nn", X1, {**nn_p, "attn_layers": 2}, "nn_tuned"),
    "nn_emb_attn1": ("nn", X1_ids, {**nn_p, "id_cols": IDS, "attn_layers": 1}, "nn_tuned"),
    "nn_c2": ("nn", X2, dict(nn_p), "nn_tuned"),
    "lgb_c2": ("lgb_binary", X2, zoo.tuned_params("lgb", best["lgb"]["params"]), "lgb_tuned"),
    "xgb_c2": ("xgb_binary", X2, zoo.tuned_params("xgb", best["xgb"]["params"]), "xgb_tuned"),
}
RES, OOFS = [], {}
for name, (recipe, X, params, ref) in EXPS.items():
    cached = OOF_DIR / f"{name}.parquet"
    if cached.exists():
        OOFS[name] = pd.read_parquet(cached)
        res = zoo_runner.metrics_by_year(OOFS[name][["year"]].reset_index(drop=True), {t: OOFS[name][f"p_{t}"].to_numpy() for t in ("win", "top2", "top3")},
                                         OOFS[name].merge(meta[["race_id", "horse_id", "finish_position"]], on=["race_id", "horse_id"])["finish_position"].to_numpy(), OOFS[name]["race_id"].to_numpy(), name)
        res.attrs["seconds"] = 0.0
        print(f"  {name:14s} (再利用) win {res['logloss_win'].mean():.5f}")
    else:
        with mlflow.start_run(run_name=f"tp-05f-{name}", tags={"notebook": "tp-05f", "git_commit": GIT_COMMIT, "variant": name, "base": ref}):
            res, OOFS[name] = zoo_runner.run_oof(name, recipe, X, meta, folds, SEEDS, THREADS, params=dict(params), out_dir=OOF_DIR)
            num = [c for c in res.columns if c not in ("model", "fold")]
            mlflow.log_params({"recipe": recipe, "n_features": X.shape[1], **{f"p_{k}": str(v) for k, v in params.items() if k != "weight"}})
            mlflow.log_metrics({f"mean_{c}": float(res[c].mean()) for c in num})
            mlflow.log_metric("seconds", res.attrs["seconds"])
        print(f"  {name:14s} win {res['logloss_win'].mean():.5f} top2 {res['logloss_top2'].mean():.5f} top3 {res['logloss_top3'].mean():.5f} ({res.attrs['seconds']:.0f}s)")
    RES.append(res)
    gc.collect()
pd.concat(RES).to_csv(OUT / "fold_metrics.csv", index=False)

In [ ]:
key = ["race_id", "horse_id"]
fin_map = meta.set_index(key)["finish_position"]
rows = []
for name, (_, _, _, ref) in EXPS.items():
    b = pd.read_parquet(OOF_DIR / f"{ref}.parquet")
    m = b.merge(OOFS[name], on=key, suffixes=("_b", "_v"))
    fin = fin_map.loc[list(zip(m["race_id"], m["horse_id"]))].to_numpy()
    out = {"variant": name, "基準": ref}
    for tgt, k in (("win", 1), ("top2", 2), ("top3", 3)):
        y = (fin <= k).astype(float)
        bb = wm.paired_bootstrap_diff(wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{tgt}_b"].to_numpy(), y), wm.per_race_loss(m["race_id"].to_numpy(), m[f"p_{tgt}_v"].to_numpy(), y), n_boot=500)
        out[f"{tgt}_base"] = wm.binary_logloss(m[f"p_{tgt}_b"].to_numpy(), y); out[f"{tgt}_new"] = wm.binary_logloss(m[f"p_{tgt}_v"].to_numpy(), y)
        out[f"Δ{tgt}"], out[f"{tgt}_hi"] = bb["diff"], bb["hi"]
    out["採用"] = bool(out["win_hi"] < 0 and out["Δtop2"] <= 0.0002 and out["Δtop3"] <= 0.0002)
    rows.append(out)
cmp_ = pd.DataFrame(rows).set_index("variant")
display(cmp_.round(5)); cmp_.to_csv(OUT / "vs_base.csv")
adopted = cmp_[cmp_["採用"]].sort_values("Δwin").index.tolist()
print("採用基準を満たした案（改善の大きい順）:", adopted)
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "adopted": adopted, "vs_base": cmp_.round(6).reset_index().to_dict("records")}, ensure_ascii=False, indent=1, default=str), encoding="utf-8")